# 08 - Değerlendirme ve raporlama
A0-A4 ablasyonlarını aynı test ayrımında karşılaştırır; açı MAE/RMSE, ağır hata AUROC/AUPRC ve risk-kapsama eğrisi üretir.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import json, pandas as pd
from physioload.metrics import classification_auc, mae, risk_coverage, rmse

results_path = env.predictions_dir / 'test_predictions.csv'
if not results_path.exists():
    raise FileNotFoundError(f'Test tahminleri gerekli: {results_path}')
results = pd.read_csv(results_path)
required = {'ablation','reference_angle_deg','predicted_angle_deg','severe_error','severe_error_probability','reliability_score'}
missing = required - set(results.columns)
if missing:
    raise ValueError(f'Eksik sütunlar: {sorted(missing)}')

In [ ]:
summary = []
for ablation, frame in results.groupby('ablation'):
    auc = classification_auc(frame.severe_error, frame.severe_error_probability)
    curve = risk_coverage((frame.reference_angle_deg - frame.predicted_angle_deg).abs(), frame.reliability_score)
    summary.append({
        'ablation': ablation,
        'angle_mae_deg': mae(frame.reference_angle_deg, frame.predicted_angle_deg),
        'angle_rmse_deg': rmse(frame.reference_angle_deg, frame.predicted_angle_deg),
        **auc, 'aurc': curve['aurc'],
    })
summary = pd.DataFrame(summary).sort_values('ablation')
summary

In [ ]:
summary_path = env.metrics_dir / f'evaluation_{env.processed_version}.csv'
summary.to_csv(summary_path, index=False)
print('saved:', summary_path)